# RLHF 完整三阶段 Pipeline

> InstructGPT / Llama2-Chat 的标准对齐流程。

## 三阶段
1. **SFT**：监督微调，用高质量对话数据 fine-tune base model
2. **RM**：训练奖励模型，用人类偏好对比数据
3. **PPO**：用 RM 奖励 + KL 罚在线训练 policy

## PPO 损失
L_total = L_PG - c_entropy * H(pi) + c_vf * L_VF - beta * KL(pi || pi_SFT)
- L_PG: clip 目标，用 importance ratio rho_t = pi/pi_old
- KL 罚：防 pi 偏离 SFT 过远
- GAE: 用 V 函数估计优势

## 数据流
SFT data -> [(x, y_chosen, y_rejected)] -> RM -> reward scores
RM + prompts -> PPO rollout -> (x, y, reward) -> update

## 考察点
- 三阶段各自的数据和损失
- KL 罚的作用（防 reward hacking）
- PPO 与 DPO 的 online/offline 区别


In [ ]:
import torch; import torch.nn as nn
from copy import deepcopy

class RLHFPipeline:
    def __init__(self, sft_model, reward_model, ref_model=None):
        self.policy = sft_model
        self.rm = reward_model
        self.ref = ref_model or deepcopy(sft_model)
        for p in self.rm.parameters(): p.requires_grad = False
        for p in self.ref.parameters(): p.requires_grad = False

    def compute_reward(self, input_ids):
        # TODO: RM 打分
        raise NotImplementedError

    def compute_kl_penalty(self, policy_logprobs, ref_logprobs):
        # TODO: KL(pi || ref) token-wise sum
        raise NotImplementedError

    def ppo_step(self, batch, epsilon=0.2, beta=0.04, entropy_coef=0.01):
        # TODO: 完整 PPO 更新 step
        # 1. 计算 old_logprobs, ref_logprobs (no_grad)
        # 2. 前向 policy 得 new_logprobs, values
        # 3. GAE 计算优势
        # 4. clip 目标 + value loss + entropy + KL loss
        raise NotImplementedError

# ===== 测试验证 =====
print("RLHF Pipeline 框架已定义")
print("核心: compute_reward + compute_kl_penalty + ppo_step")
print("\u2139 完整流程需真实模型和偏好数据对")
